**BASE DE DADOS HISTORICOS**

 -- CAPTURA INFORMAÇÕES HISTÓRICAS DA B3, BANCO CENTRAL, TESOURO DIRETO

In [7]:
!pip install pandas

  Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ------- -------------------------------- 1.8/9.7 MB 10.9 MB/s eta 0:00:01
   ------------------ --------------------- 4.5/9.7 MB 11.3 MB/s eta 0:00:01
   --------------------------- ------------ 6.8/9.7 MB 11.3 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.7 MB 11.3 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 10.9 MB/s  0:00:00
Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl (12.3 MB)

   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- ------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
!pip install pyarrow

   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   - -------------------------------------- 1.3/27.5 MB 9.9 MB/s eta 0:00:03
   ----- ---------------------------------- 3.7/27.5 MB 11.1 MB/s eta 0:00:03
   -------- ------------------------------- 6.0/27.5 MB 11.3 MB/s eta 0:00:02
   ------------ --------------------------- 8.7/27.5 MB 11.4 MB/s eta 0:00:02
   --------------- ------------------------ 11.0/27.5 MB 11.5 MB/s eta 0:00:02
   ------------------- -------------------- 13.4/27.5 MB 11.5 MB/s eta 0:00:02
   ---------------------- ----------------- 15.7/27.5 MB 11.5 MB/s eta 0:00:02
   -------------------------- ------------- 18.1/27.5 MB 11.5 MB/s eta 0:00:01
   ----------------------------- ---------- 20.4/27.5 MB 11.5 MB/s eta 0:00:01
   --------------------------------- ------ 22.8/27.5 MB 11.6 MB/s eta 0:00:01
   ------------------------------------ --- 25.2/27.5 MB 11.6 MB/s eta 0:00:01
   ---------------------------------------  27.5/27.5 MB 11.6 MB/s


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
!pip install yfinance

  Using cached multitasking-0.0.12-py3-none-any.whl
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached frozendict-2.4.7-py3-none-any.whl.metadata (23 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached websockets-16.0-cp313-cp313-win_amd64.whl.metadata (7.0 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 10.3 MB/s  0:00:00
Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl (1


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
import requests
import zipfile
import pandas as pd
from io import BytesIO, StringIO
import glob
import time

BASE_DIR = "data"
os.makedirs(BASE_DIR, exist_ok=True)

# =========================
# SAVE PARQUET (SEM BUG)
# =========================
def save_parquet(df, path):
    import pyarrow as pa
    import pyarrow.parquet as pq

    df = df.reset_index(drop=True)
    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, path)


# =========================
# 1. DOWNLOAD B3
# =========================
def download_b3(year):
    print(f"Baixando B3 {year}...")

    url = f"https://bvmf.bmfbovespa.com.br/InstDados/SerHist/COTAHIST_A{year}.ZIP"

    r = requests.get(url)
    if r.status_code != 200:
        print(f"Erro no ano {year}")
        return

    z = zipfile.ZipFile(BytesIO(r.content))
    file_name = z.namelist()[0]

    with z.open(file_name) as f:
        content = f.read().decode("latin-1")

    df = pd.read_fwf(
        StringIO(content),
        widths=[
            2,8,2,12,3,12,10,3,4,13,13,13,13,13,13,13,
            5,18,18,13,1,8,7,13,12,3
        ],
        header=None
    )

    df.columns = [
        "tipo_registro","data","cod_bdi","ticker","tipo_mercado",
        "nome","espec","prazo","moeda",
        "preco_abertura","preco_max","preco_min","preco_medio",
        "preco_fechamento","preco_melhor_compra","preco_melhor_venda",
        "negocios","quantidade","volume","preco_exercicio",
        "indicador_correcao","data_vencimento","fator_cotacao",
        "preco_exercicio_pontos","codigo_isin","num_distribuicao"
    ]

    # filtro correto
    df["tipo_registro"] = df["tipo_registro"].astype(str).str.zfill(2)
    df = df[df["tipo_registro"] == "01"]

    # datas
    df["data"] = pd.to_datetime(df["data"], format="%Y%m%d", errors="coerce")
    df = df.dropna(subset=["data"])

    # ticker
    df["ticker"] = df["ticker"].str.strip()
    df = df[df["ticker"] != ""]

    # preços
    for col in ["preco_abertura","preco_max","preco_min","preco_fechamento"]:
        df[col] = pd.to_numeric(df[col], errors="coerce") / 100

    df["volume"] = pd.to_numeric(df["volume"], errors="coerce")

    df = df[df["volume"] > 0]

    df = df[[
        "data","ticker",
        "preco_abertura","preco_max","preco_min",
        "preco_fechamento","volume"
    ]]

    save_parquet(df, f"{BASE_DIR}/b3_{year}.parquet")

    print(f"OK {year}")


# =========================
# 2. SELIC HISTÓRICA
# =========================
def get_selic():
    print("Baixando Selic...")

    url1 = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.11/dados?formato=csv&dataInicial=01/01/2010&dataFinal=01/01/2020"
    url2 = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.11/dados?formato=csv&dataInicial=02/01/2020&dataFinal=01/01/2026"

    r1 = requests.get(url1)
    r2 = requests.get(url2)

    df1 = pd.read_csv(StringIO(r1.text), sep=";")
    df2 = pd.read_csv(StringIO(r2.text), sep=";")

    df = pd.concat([df1, df2])
    df.columns = ["data", "selic"]

    df["data"] = pd.to_datetime(df["data"], dayfirst=True, errors="coerce")

    df["selic"] = (
        df["selic"]
        .astype(str)
        .str.replace(",", ".", regex=False)
    )

    df["selic"] = pd.to_numeric(df["selic"], errors="coerce")

    df = df.dropna().sort_values("data")

    save_parquet(df, f"{BASE_DIR}/selic.parquet")

    print("Selic OK")


# =========================
# 3. CARREGAR B3 COMPLETA
# =========================
def load_b3():
    import pyarrow.parquet as pq

    files = glob.glob("data/b3_*.parquet")

    dfs = []
    for f in files:
        table = pq.read_table(f)
        dfs.append(table.to_pandas())

    df = pd.concat(dfs, ignore_index=True)

    df = df.sort_values(["ticker", "data"])

    return df


# =========================
# 4. FEATURES
# =========================
def create_features(df):

    df = df.sort_values(["ticker", "data"])

    # retorno
    df["return_1d"] = df.groupby("ticker")["preco_fechamento"].pct_change()

    # médias móveis
    df["ma_7"] = df.groupby("ticker")["preco_fechamento"].transform(lambda x: x.rolling(7).mean())
    df["ma_30"] = df.groupby("ticker")["preco_fechamento"].transform(lambda x: x.rolling(30).mean())

    # volatilidade
    df["vol_7"] = df.groupby("ticker")["return_1d"].transform(lambda x: x.rolling(7).std())

    return df


# =========================
# 5. MERGE SELIC
# =========================
def merge_selic(df):
    import pyarrow.parquet as pq

    selic = pq.read_table("data/selic.parquet").to_pandas()

    # garantir ordenação
    df = df.sort_values("data")
    selic = selic.sort_values("data")

    df = df.merge(selic, on="data", how="left")

    # forward fill correto
    df["selic"] = df["selic"].ffill()

    return df


# =========================
# 6. MERCADO GERAL
# =========================
def add_market(df):
    market = df.groupby("data")["preco_fechamento"].mean().reset_index()
    market.columns = ["data", "market_mean"]

    df = df.merge(market, on="data", how="left")

    return df


# =========================
# 7. TARGET
# =========================
def create_target(df):
    df["target"] = df.groupby("ticker")["preco_fechamento"].pct_change().shift(-1)
    return df


# =========================
# 8. DATASET FINAL
# =========================
def build_dataset():
    print("Construindo dataset final...")

    df = load_b3()

    df = create_features(df)

    df = merge_selic(df)

    df = add_market(df)

    df = create_target(df)

    df = df.dropna()

    save_parquet(df, "data/dataset_final.parquet")

    print("DATASET PRONTO 🚀")


# =========================
# EXECUÇÃO
# =========================
if __name__ == "__main__":

    # 🔥 1. Baixar histórico B3
    for year in range(2010, 2025):
        download_b3(year)

    # 🔥 2. Selic
    get_selic()

    # 🔥 3. Dataset final
    build_dataset()

Baixando B3 2010...
OK 2010
Baixando B3 2011...
OK 2011
Baixando B3 2012...
OK 2012
Baixando B3 2013...
OK 2013
Baixando B3 2014...
OK 2014
Baixando B3 2015...
OK 2015
Baixando B3 2016...
OK 2016
Baixando B3 2017...
OK 2017
Baixando B3 2018...
OK 2018
Baixando B3 2019...
OK 2019
Baixando B3 2020...
OK 2020
Baixando B3 2021...
OK 2021
Baixando B3 2022...
OK 2022
Baixando B3 2023...
OK 2023
Baixando B3 2024...
OK 2024
Baixando Selic...
Selic OK
Construindo dataset final...
DATASET PRONTO 🚀


In [10]:
import pandas as pd
df = pd.read_parquet("E:\\TCC\\d_functions\\data\\dataset_final.parquet", engine="pyarrow")
df.head()

,data,ticker,preco_abertura,preco_max,preco_min,preco_fechamento,volume,return_1d,ma_7,ma_30,vol_7,selic,market_mean,target
0,2010-01-06,PETR4T,38.13,38.14,38.13,38.14,1906970.0,0.000000,38.262857,37.742667,0.016087,0.032927,55.503192,0.014158
1,2010-01-06,PETR4T,38.67,38.68,38.67,38.68,1160270.0,0.014158,38.412857,37.794667,0.015906,0.032927,55.503192,0.011117
2,2010-01-06,PETR4T,38.81,39.27,38.81,39.11,22664010.0,0.011117,38.555714,37.854000,0.015768,0.032927,55.503192,-0.042700
3,2010-01-06,PETR4T,37.20,37.56,37.20,37.44,30686170.0,-0.042700,38.347143,37.857333,0.021581,0.032927,55.503192,-0.004274
4,2010-01-06,PETR4T,37.36,37.37,37.36,37.37,1868250.0,0.002414,38.022857,37.846000,0.019076,0.032927,55.503192,0.004817
